# 01_build_sft_dataset (span-schema 版)

为 PII redaction 构造 SFT 数据。

## Schema 设计

- **输出格式**:模型输出带 XML 风格标签的文本,例如 `<pii type="PERSON">Mark Chen</pii> lives at <pii type="ADDRESS">Sydney</pii>`
- **后处理**:`redaction_utils.parse_annotated()` 把标签文本解析成 `[{start, end, type, value}]` 列表
- **为什么不让模型直接输出 offset**:LLM 对字符计数不擅长,差 1-2 位很常见;让模型输出带标签的文本更自然,parse 一步得到精确 offset

## 相对 value-schema 旧版的关键改动

- 保留所有 span(不做 `(type, value)` 去重)—— redaction 需要定位每次出现
- 重叠 span 按 confidence 最高的保留(`redaction_utils.annotate_text` 自动处理)
- 验证 `label.value == text[start:end]`,不一致的跳过(数据清洗)
- 额外导出 test 集的 ground-truth spans 给评估 notebook 用

## 依赖

需要同目录下的 `redaction_utils.py`。

In [1]:
import json
import os
import random
import sys
from collections import Counter
from sklearn.model_selection import train_test_split

sys.path.insert(0, ".")
from redaction_utils import annotate_text, parse_annotated

## 1. 路径与常量

In [2]:
RAW_DATA_PATH = "../data/raw/au_pii_19000_final.json"
SAVE_DIR = "../data/processed"
SEED = 42

random.seed(SEED)
os.makedirs(SAVE_DIR, exist_ok=True)

assert os.path.exists(RAW_DATA_PATH), f"找不到 {RAW_DATA_PATH}"
print("输入:", RAW_DATA_PATH)
print("输出:", SAVE_DIR)

输入: ../data/raw/au_pii_19000_final.json
输出: ../data/processed


## 2. 读取原始数据

In [3]:
with open(RAW_DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

records = data["records"]
print("总记录数:", len(records))
print("版本:", data.get("version"))

总记录数: 19000
版本: 2.0-final


## 3. 目标类型 (16 类)

In [4]:
TARGET_LABELS = [
    "PERSON", "ADDRESS", "DATE_OF_BIRTH",
    "EMAIL_ADDRESS", "AU_PHONE",
    "AU_TFN", "AU_PASSPORT", "AU_DRIVERS_LICENCE",
    "STUDENT_ID", "MEDICARE_NUMBER",
    "AU_BANK_ACCOUNT", "BSB", "PAYMENT_CARD_NUMBER",
    "IP_ADDRESS", "VEHICLE_REGO", "SALARY",
]
TARGET_SET = set(TARGET_LABELS)
print("使用标签数:", len(TARGET_LABELS))

使用标签数: 16


## 4. System prompt

In [5]:
SYSTEM_PROMPT = (
    "You are a PII annotator for Australian context.\n"
    "You receive a piece of text and output the SAME text with PII entities wrapped in XML-style tags.\n"
    "\n"
    "Tag format:\n"
    '  <pii type="TYPE">VALUE</pii>\n'
    "\n"
    "Supported types (only wrap spans of these types; ignore all other sensitive attributes):\n"
    + "\n".join(f"  - {t}" for t in TARGET_LABELS) + "\n"
    "\n"
    "Strict rules:\n"
    "  - Preserve every character of the input exactly: whitespace, punctuation, case, newlines.\n"
    "  - Wrap EVERY occurrence of supported PII. Do not deduplicate.\n"
    "  - Do not nest tags. Do not output explanations, comments, or markdown fences.\n"
    "  - If no supported PII is present, return the input text unchanged.\n"
)
print(SYSTEM_PROMPT)

You are a PII annotator for Australian context.
You receive a piece of text and output the SAME text with PII entities wrapped in XML-style tags.

Tag format:
  <pii type="TYPE">VALUE</pii>

Supported types (only wrap spans of these types; ignore all other sensitive attributes):
  - PERSON
  - ADDRESS
  - DATE_OF_BIRTH
  - EMAIL_ADDRESS
  - AU_PHONE
  - AU_TFN
  - AU_PASSPORT
  - AU_DRIVERS_LICENCE
  - STUDENT_ID
  - MEDICARE_NUMBER
  - AU_BANK_ACCOUNT
  - BSB
  - PAYMENT_CARD_NUMBER
  - IP_ADDRESS
  - VEHICLE_REGO
  - SALARY

Strict rules:
  - Preserve every character of the input exactly: whitespace, punctuation, case, newlines.
  - Wrap EVERY occurrence of supported PII. Do not deduplicate.
  - Do not nest tags. Do not output explanations, comments, or markdown fences.
  - If no supported PII is present, return the input text unchanged.



## 5. 构造单条样本

In [6]:
def build_sample(text, labels):
    in_target = [l for l in labels if l["type"] in TARGET_SET]
    out_of_target = [l for l in labels if l["type"] not in TARGET_SET]

    annotated, used = annotate_text(text, in_target, target_types=TARGET_SET)
    dropped_overlaps = len(in_target) - len(used)

    sample = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Annotate PII in the following text:\n\n{text}"},
            {"role": "assistant", "content": annotated},
        ]
    }
    return sample, used, {"out_of_target": out_of_target, "overlaps_dropped": dropped_overlaps}

## 6. 单条 round-trip 验证

In [7]:
rec = records[0]
text = rec["positive_sample"]["text"]
labels = rec["positive_sample"]["labels"]

sample, used_labels, stats = build_sample(text, labels)
annotated = sample["messages"][-1]["content"]
plain, parsed_spans = parse_annotated(annotated)

print(f"原始长度:     {len(text)}")
print(f"解析后长度:   {len(plain)}")
print(f"一致:         {plain == text}")
print(f"原始 labels:  {len(labels)}")
print(f"过滤到 16 类: {len(labels) - len(stats['out_of_target'])}")
print(f"重叠裁掉:     {stats['overlaps_dropped']}")
print(f"最终标记:     {len(used_labels)}")
print(f"parse 得到:   {len(parsed_spans)}")
assert plain == text
assert len(parsed_spans) == len(used_labels)
for u, p in zip(sorted(used_labels, key=lambda x: x['start']),
                sorted(parsed_spans, key=lambda x: x['start'])):
    assert u['start'] == p['start']
    assert u['end'] == p['end']
    assert u['type'] == p['type']
print('\n✓ round-trip 完全一致')
print('\nANNOTATED 前 600 字:')
print(annotated[:600])

原始长度:     1272
解析后长度:   1272
一致:         True
原始 labels:  9
过滤到 16 类: 4
重叠裁掉:     0
最终标记:     4
parse 得到:   4

✓ round-trip 完全一致

ANNOTATED 前 600 字:
---------- Forwarded message ---------
From: Marcus Holloway <marcus.holloway@rivertoninstitute.com.au>
Date: Thu, 17 Feb 2025 at 14:11
Subject: Re: Re: case review — Nakamura
To: casework@rivertoninstitute.com.au

Just a quick update on this one ahead of the review meeting. The client's marital status has recently been updated to widowed, and I'd flag that there's an outstanding HECS-HELP debt: $19,463 that the financial team will need to factor in when assessing eligibility.

On the service side, the file shows the individual is an ADF member — Australian Army, which may well affect which pr


## 7. 批量构建

In [8]:
positives = []
negatives = []
type_counter = Counter()
out_of_target_counter = Counter()
overlaps_dropped_total = 0
invalid_records = 0
pos_empty_after_filter = 0

for rec in records:
    text = rec['positive_sample']['text']
    labels = rec['positive_sample']['labels']

    try:
        sample, used, stats = build_sample(text, labels)
    except Exception as e:
        invalid_records += 1
        continue

    for ent in used:
        type_counter[ent['type']] += 1
    for ent in stats['out_of_target']:
        out_of_target_counter[ent['type']] += 1
    overlaps_dropped_total += stats['overlaps_dropped']

    if len(used) == 0:
        pos_empty_after_filter += 1

    positives.append(sample)

    for neg in rec.get('hard_negatives', []):
        neg_sample, _, _ = build_sample(neg['text'], [])
        negatives.append(neg_sample)

print(f'正样本:                {len(positives)}')
print(f'  过滤后为空的:       {pos_empty_after_filter}  ({pos_empty_after_filter/max(1,len(positives))*100:.1f}%)')
print(f'负样本:                {len(negatives)}')
print(f'跳过的 invalid record: {invalid_records}')
print(f'重叠裁掉的 span 总数:  {overlaps_dropped_total}')
print('\n每类被标记的次数:')
for t in TARGET_LABELS:
    print(f'  {t:<22s} {type_counter[t]}')
print('\n不在 16 类被丢弃的 (top 20):')
for t, c in out_of_target_counter.most_common(20):
    print(f'  {t:<30s} {c}')

正样本:                19000
  过滤后为空的:       0  (0.0%)
负样本:                75991
跳过的 invalid record: 0
重叠裁掉的 span 总数:  0

每类被标记的次数:
  PERSON                 19000
  ADDRESS                11582
  DATE_OF_BIRTH          10817
  EMAIL_ADDRESS          5146
  AU_PHONE               5071
  AU_TFN                 2500
  AU_PASSPORT            3400
  AU_DRIVERS_LICENCE     2500
  STUDENT_ID             3200
  MEDICARE_NUMBER        1500
  AU_BANK_ACCOUNT        1800
  BSB                    1800
  PAYMENT_CARD_NUMBER    1698
  IP_ADDRESS             900
  VEHICLE_REGO           900
  SALARY                 3000

不在 16 类被丢弃的 (top 20):
  CREDIT_CARD_EXPIRY             1698
  IHI                            1500
  MEDICARE_EXPIRY                1500
  WAM_SCORE                      1500
  SANCTIONS                      1500
  SCHOLARSHIP                    1500
  SUBJECT_RESULTS                1500
  WORK_EMAIL                     1200
  WORK_PHONE                     1200
  EMPLOYMENT_INFORMATION 

## 8. 空正样本策略

过滤后为空的正样本和 hard negative 性质类似(assistant=原文)。如果比例过高可以排除:

In [9]:
EXCLUDE_EMPTY_POSITIVES = False  # True 会排除

if EXCLUDE_EMPTY_POSITIVES:
    before = len(positives)
    positives = [s for s in positives if '<pii type=' in s['messages'][-1]['content']]
    print(f'排除空正样本: {before} -> {len(positives)}')
else:
    print('保留空正样本')

保留空正样本


## 9. 正负样本比例

In [10]:
NEG_POS_RATIO = 1.0
n_neg_keep = int(len(positives) * NEG_POS_RATIO)
random.shuffle(negatives)
negatives_sampled = negatives[:n_neg_keep]
print(f'1:{NEG_POS_RATIO}  pos={len(positives)}  neg={len(negatives_sampled)}')

1:1.0  pos=19000  neg=19000


## 10. train / dev / test 分层切分

In [11]:
def split_three_way(items, seed=SEED):
    train, temp = train_test_split(items, test_size=0.2, random_state=seed, shuffle=True)
    dev, test = train_test_split(temp, test_size=0.5, random_state=seed, shuffle=True)
    return train, dev, test

pos_tr, pos_dv, pos_te = split_three_way(positives)
neg_tr, neg_dv, neg_te = split_three_way(negatives_sampled)

train_data = pos_tr + neg_tr
dev_data = pos_dv + neg_dv
test_data = pos_te + neg_te
random.shuffle(train_data)
random.shuffle(dev_data)
random.shuffle(test_data)

print(f'train: {len(train_data)}  (pos={len(pos_tr)} neg={len(neg_tr)})')
print(f'dev:   {len(dev_data)}  (pos={len(pos_dv)} neg={len(neg_dv)})')
print(f'test:  {len(test_data)}  (pos={len(pos_te)} neg={len(neg_te)})')

train: 30400  (pos=15200 neg=15200)
dev:   3800  (pos=1900 neg=1900)
test:  3800  (pos=1900 neg=1900)


## 11. 长度分布

In [12]:
def sample_char_len(sample):
    return sum(len(m['content']) for m in sample['messages'])

lens = sorted(sample_char_len(s) for s in train_data)
n = len(lens)
print(f'char - mean={sum(lens)/n:.0f}  p50={lens[n//2]}  p95={lens[int(n*0.95)]}  p99={lens[int(n*0.99)]}  max={lens[-1]}')

char - mean=1616  p50=1289  p95=3127  p99=3689  max=4403


## 12. 抽样检查

In [13]:
pos_ex = next(s for s in train_data if '<pii type=' in s['messages'][-1]['content'])
print('=== 正样本 ===')
print('[user]     ', pos_ex['messages'][1]['content'][:300], '...')
print('[assistant]', pos_ex['messages'][2]['content'][:600])

=== 正样本 ===
[user]      Annotate PII in the following text:

[VOICEMAIL TRANSCRIPT — 10/06/2025 15:07 — auto-generated, unverified]

Hey, it's M. Reyes here, sorry to leave another message. Just wanted to confirm a few things on file. The card I used for the initial payment was 5376 8456 3106 7454, and that one expires 07/ ...
[assistant] [VOICEMAIL TRANSCRIPT — 10/06/2025 15:07 — auto-generated, unverified]

Hey, it's <pii type="PERSON">M. Reyes</pii> here, sorry to leave another message. Just wanted to confirm a few things on file. The card I used for the initial payment was <pii type="PAYMENT_CARD_NUMBER">5376 8456 3106 7454</pii>, and that one expires 07/2029. The bank account details are with Bendigo Bank — the BSB is <pii type="BSB">082-300</pii> and the account number is <pii type="AU_BANK_ACCOUNT">109178871</pii>. The mailing address we have on record is <pii type="ADDRESS">127 Elizabeth Cres, Albany WA 6100</pii>, so h


In [14]:
neg_ex = next(s for s in train_data if '<pii type=' not in s['messages'][-1]['content'])
print('=== 负样本 ===')
print('[user]     ', neg_ex['messages'][1]['content'])
print('[assistant]', neg_ex['messages'][2]['content'])

=== 负样本 ===
[user]      Annotate PII in the following text:

The internal database key generated for this submission is 988900185; please keep it for your records.
[assistant] The internal database key generated for this submission is 988900185; please keep it for your records.


## 13. 额外保存 test ground truth

评估 notebook 会加载这个文件,和模型输出 parse 出的 spans 对比。

In [15]:
def sample_to_gt(sample):
    user_content = sample['messages'][1]['content']
    prefix = 'Annotate PII in the following text:\n\n'
    assert user_content.startswith(prefix)
    raw_text = user_content[len(prefix):]
    annotated = sample['messages'][-1]['content']
    plain, spans = parse_annotated(annotated)
    assert plain == raw_text
    return {'text': raw_text, 'spans': spans}

test_gt = [sample_to_gt(s) for s in test_data]
print(f'test ground truth: {len(test_gt)} 条')
print('\n示例:')
print(json.dumps(test_gt[0], ensure_ascii=False, indent=2)[:800])

test ground truth: 3800 条

示例:
{
  "text": "To the HR Director,\n\nThis correspondence is intended to place on record the particulars set out below.\n\nA driver licence, TAS: 667104, was also presented as secondary identification. For completeness, the tax records associated with this employment have been lodged under 210 177 639.\n\nIdentity was established by travel document L 7252490. The writer, O'Brien, Ingrid, of 366 Pitt Way, Bunbury WA 6150 born 13 June 1990.\n\nI would be grateful for a substantive response within 14 days.\n\nRegards,\nIngrid",
  "spans": [
    {
      "start": 122,
      "end": 133,
      "type": "AU_DRIVERS_LICENCE",
      "value": "TAS: 667104"
    },
    {
      "start": 272,
      "end": 283,
      "type": "AU_TFN",
      "value": "210 177 639"
    },
    {
      "start": 330,
      "end":


## 14. 保存

In [16]:
def save_jsonl(path, items):
    with open(path, 'w', encoding='utf-8') as f:
        for it in items:
            f.write(json.dumps(it, ensure_ascii=False) + '\n')
    print(f'saved: {path}  ({len(items)} lines)')

save_jsonl(os.path.join(SAVE_DIR, 'qwen_sft_train.jsonl'), train_data)
save_jsonl(os.path.join(SAVE_DIR, 'qwen_sft_dev.jsonl'), dev_data)
save_jsonl(os.path.join(SAVE_DIR, 'qwen_sft_test.jsonl'), test_data)
save_jsonl(os.path.join(SAVE_DIR, 'test_ground_truth.jsonl'), test_gt)

saved: ../data/processed/qwen_sft_train.jsonl  (30400 lines)
saved: ../data/processed/qwen_sft_dev.jsonl  (3800 lines)
saved: ../data/processed/qwen_sft_test.jsonl  (3800 lines)
saved: ../data/processed/test_ground_truth.jsonl  (3800 lines)


In [17]:
meta = {
    'schema': 'span-tagged',
    'tag_format': '<pii type="TYPE">VALUE</pii>',
    'target_labels': TARGET_LABELS,
    'system_prompt': SYSTEM_PROMPT,
    'seed': SEED,
    'neg_pos_ratio': NEG_POS_RATIO,
    'train_size': len(train_data),
    'dev_size': len(dev_data),
    'test_size': len(test_data),
    'type_counts': dict(type_counter),
    'out_of_target_top20': dict(out_of_target_counter.most_common(20)),
    'overlaps_dropped_total': overlaps_dropped_total,
    'pos_empty_after_filter': pos_empty_after_filter,
    'excluded_empty_positives': EXCLUDE_EMPTY_POSITIVES,
}
with open(os.path.join(SAVE_DIR, 'meta.json'), 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print('saved meta.json')

saved meta.json
